<a href="https://colab.research.google.com/github/krvax/ollama-sre-workspace/blob/master/python-fundamentals/04_archivos_logs_sre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 04: Archivos + Logs para SRE

**Tema:** open(), read, write, with, try/except, listas, diccionarios

**Contexto SRE:** Tienes un archivo de log de un servicio. Necesitas leerlo, parsear cada línea, clasificar por severidad, y generar un reporte resumen.

**Instrucciones:**
1. Ejecuta la celda de creación del archivo de log
2. Completa cada función según las instrucciones
3. Verifica el reporte final

## Paso 0: Crear el archivo de log de ejemplo
Ejecuta esta celda una vez para generar el archivo de prueba.

In [2]:
log_lines = [
    "2026-05-23 08:01:12 INFO api-gateway: Request received /health",
    "2026-05-23 08:01:13 INFO auth-service: Token validated",
    "2026-05-23 08:01:15 ERROR db-primary: Connection timeout after 30s",
    "2026-05-23 08:01:16 WARN cache-redis: Memory usage at 85%",
    "2026-05-23 08:01:18 ERROR payment-svc: Transaction failed - insufficient funds",
    "2026-05-23 08:01:20 INFO api-gateway: Request received /users",
    "2026-05-23 08:01:22 WARN db-primary: Slow query detected (2.3s)",
    "2026-05-23 08:01:25 ERROR auth-service: Invalid token - expired",
    "2026-05-23 08:01:27 INFO cache-redis: Cache hit ratio 94%",
    "2026-05-23 08:01:30 ERROR db-primary: Connection timeout after 30s",
    "2026-05-23 08:01:32 WARN payment-svc: Retry attempt 2/3",
    "2026-05-23 08:01:35 INFO api-gateway: Health check OK",
]

with open('service.log', 'w') as f:
    for line in log_lines:
        f.write(line + '\n')

print('Archivo service.log creado.')

Archivo service.log creado.


## 1. Función `leer_log`

Crea una función que reciba un nombre de archivo (string).
- Usa `with open(...) as f` para abrir el archivo en modo lectura
- Lee todas las líneas y retorna una lista de strings (sin el `\n` al final)
- Si el archivo no existe, atrapa `FileNotFoundError` y retorna una lista vacía
- Imprime un mensaje de error antes de retornar la lista vacía

**Pistas:**
- `f.readlines()` te da todas las líneas como lista
- `.strip()` quita espacios y `\n` de un string
- El `try` debe envolver al `open` (ahí es donde puede fallar)

In [3]:
# Tu código aquí
def leer_log(file):
    try:
      with open(file,"r") as f:
        return f.readlines()
    except FileNotFoundError:
      print(f"Archivo {file} vacío.")
      return []



In [4]:
# Prueba leer_log
lineas = leer_log('service.log')
print(f'Líneas leídas: {len(lineas)}')  # Debería dar 12

lineas2 = leer_log('no_existe.log')
print(f'Líneas leídas: {len(lineas2)}')  # Debería dar 0 + mensaje de error

Líneas leídas: 12
Archivo no_existe.log vacío.
Líneas leídas: 0


## 2. Función `parsear_linea`

Crea una función que reciba una línea de log (string) y retorne un diccionario.

Ejemplo de entrada:
```
"2026-05-23 08:01:12 INFO api-gateway: Request received /health"
```

Debe retornar:
```python
{
    'timestamp': '2026-05-23 08:01:12',
    'severity': 'INFO',
    'service': 'api-gateway',
    'message': 'Request received /health'
}
```

**Pistas:**
- `.split(' ', 4)` separa en máximo 5 partes (fecha, hora, severidad, servicio:, mensaje)
- El servicio tiene un `:` al final que debes quitar (usa slicing: `texto[:-1]`)
- Si la línea no tiene 5 partes, lanza `raise ValueError("Formato inválido")`

In [5]:
# Tu código aquí
def parsear_linea(log_string):
  log_list  = log_string.split(' ', 4)
  if len(log_list) != 5:
    raise ValueError("Formato inválido")
  else:
    log = {
      'timestamp': log_list[0] + " " + log_list[1],
      'severity': log_list[2],
      'service': log_list[3][:-1],
      'message': log_list[4]
    }
    return log


In [9]:
'''
log_string = "2026-05-23 08:01:12 INFO api-gateway: Request received /health"
log_list  = log_string.split(' ', 4)
log = {
  'timestamp': log_list[0] + " " + log_list[1],
  'severity': log_list[2],
  'service': log_list[3][:-1],
  'message': log_list[4]
}
log
'''

{'timestamp': '2026-05-23 08:01:12',
 'severity': 'INFO',
 'service': 'api-gateway',
 'message': 'Request received /health'}

In [6]:
# Prueba parsear_linea
resultado = parsear_linea('2026-05-23 08:01:12 INFO api-gateway: Request received /health')
print(resultado)
# Debería imprimir: {'timestamp': '2026-05-23 08:01:12', 'severity': 'INFO', 'service': 'api-gateway', 'message': 'Request received /health'}

{'timestamp': '2026-05-23 08:01:12', 'severity': 'INFO', 'service': 'api-gateway', 'message': 'Request received /health'}


## 3. Función `generar_reporte`

Crea una función que reciba la lista de líneas y genere un reporte.
- Usa un diccionario para contar: `{'ERROR': 0, 'WARN': 0, 'INFO': 0}`
- Itera sobre cada línea y llama a `parsear_linea` dentro de un `try/except`
- Si `parsear_linea` falla (ValueError), imprime el error y continúa con la siguiente
- Al final imprime:
  - `=== REPORTE DE LOG ===`
  - `Total líneas: X`
  - `ERROR: X | WARN: X | INFO: X`

**BONUS:** También imprime los servicios que tuvieron ERROR (sin repetir).

**Pistas:**
- Un `set()` no permite duplicados — útil para servicios con error
- `sum(diccionario.values())` te da el total de todos los contadores

In [7]:
# Tu código aquí
def generar_reporte(lineas):
    contador = {'ERROR': 0, 'WARN': 0, 'INFO': 0}
    for linea in lineas:
      try:
        registro = parsear_linea(linea)
        severity = registro["severity"]
        if severity in contador:
          contador[severity] += 1
      except ValueError:
        print("No severity message found.")
    print("=== REPORTE DE LOG ===")
    print(f"Total líneas: {sum(contador.values())}")
    print(f"ERROR: {contador['ERROR']} | WARN: {contador['WARN']} | INFO: {contador['INFO']}")




## 4. Ejecutar todo junto

In [8]:
# Leer y generar reporte del archivo real
lineas = leer_log('service.log')
generar_reporte(lineas)

=== REPORTE DE LOG ===
Total líneas: 12
ERROR: 4 | WARN: 3 | INFO: 5


In [9]:
# Probar con archivo que no existe
lineas_no_existe = leer_log('no_existe.log')
generar_reporte(lineas_no_existe)

Archivo no_existe.log vacío.
=== REPORTE DE LOG ===
Total líneas: 0
ERROR: 0 | WARN: 0 | INFO: 0


## ¿Por qué esto importa en SRE?

En producción, parsear logs es el pan de cada día:
- Identificar patrones de error
- Contar frecuencia de problemas
- Generar reportes para post-mortems
- Automatizar alertas basadas en severidad

Este ejercicio simula lo que harías con un script rápido cuando no tienes acceso a Splunk/Datadog y necesitas analizar un log manualmente.